<h1 style="font-size:24px;">Cluster Nmuber</h1>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_X_y
from itertools import combinations

# ----------------------------
# Load data
# ----------------------------
pre_path = "path/to/time_series_list_24_preCondRest.npy"
post_path = "path/to/time_series_list_24_postCondRest.npy"

time_series_list_pre = list(np.load(pre_path, allow_pickle=True))    # list of arrays: (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))  # list of arrays: (T_post_i, 24)

if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, \
        f"pre[{i}] shape={ts_pre.shape}, expected (*, 24)"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, \
        f"post[{i}] shape={ts_post.shape}, expected (*, 24)"

# ----------------------------
# Preprocessing and stacking all frames
# ----------------------------
# Z-score options:
#   'time'  : z-score each ROI across time within each subject
#   'frame' : z-score each frame across ROIs within each subject
ZSCORE_MODE = 'time'   # choose from: 'time' or 'frame'

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    return np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)

X_list = []

# Pre
for ts in time_series_list_pre:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == 'time' else 1)
    X_list.append(ts)

# Post
for ts in time_series_list_post:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == 'time' else 1)
    X_list.append(ts)

if len(X_list) == 0:
    raise ValueError("No valid time series were found for CAP analysis.")

# Final frame matrix
X_cap = np.vstack(X_list)   # shape: (N_frames_kept, 24)
print("All frames stacked:", X_cap.shape)

# ----------------------------
# Ray-Turi index
# ----------------------------
def ray_turi(X, labels):
    X, labels = check_X_y(X, labels)
    le = LabelEncoder()
    labels = le.fit_transform(labels)

    n_samples, _ = X.shape
    n_labels = len(le.classes_)

    if n_labels < 2:
        raise ValueError("Number of distinct labels must be at least 2.")
    if n_samples < n_labels:
        raise ValueError("Number of samples must be >= number of labels.")

    intra_disp = 0.0
    cluster_means = []

    for k in range(n_labels):
        cluster_k = X[labels == k]
        mean_k = cluster_k.mean(axis=0)
        intra_disp += ((cluster_k - mean_k) ** 2).sum()
        cluster_means.append(mean_k)

    # Minimum squared distance between cluster centroids
    min_cluster_mean_diff = min(
        max(1e-10, ((mi - mj) ** 2).sum())
        for mi, mj in combinations(cluster_means, 2)
    )

    return intra_disp / (min_cluster_mean_diff * n_samples)

# ----------------------------
# Search over k and plot
# ----------------------------
range_n_clusters = range(2, 15)
rt_scores = []

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X_cap)
    score = ray_turi(X_cap, labels)
    rt_scores.append(score)
    print(f"k={k}, Ray-Turi index={score:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(range_n_clusters), rt_scores, marker='o')
ax.set_xlabel("Number of clusters (k)", fontsize=12)
ax.set_ylabel("Ray-Turi Index (lower is better)", fontsize=12)
ax.set_title("Ray-Turi vs. k (CAP on scrubbed frames)", fontsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks(list(range_n_clusters))

plt.tight_layout()
plt.savefig("ray_turi_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# 1) Load data and preprocess
# =========================
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score

pre_path = "path/to/time_series_list_24_preCondRest.npy"
post_path = "path/to/time_series_list_24_postCondRest.npy"

time_series_list_pre = list(np.load(pre_path, allow_pickle=True))    # list of arrays: (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))  # list of arrays: (T_post_i, 24)

if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

# Check that each subject-level time series has 24 ROI columns
for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, \
        f"pre[{i}] shape={ts_pre.shape}, expected (*, 24)"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, \
        f"post[{i}] shape={ts_post.shape}, expected (*, 24)"

# Z-score options:
#   'time'  : z-score each ROI across time within each subject
#   'frame' : z-score each frame across ROIs within each subject
ZSCORE_MODE = "time"   # choose from: "time" or "frame"

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    return np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)

X_list = []

for ts in time_series_list_pre:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == "time" else 1)
    X_list.append(ts)

for ts in time_series_list_post:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == "time" else 1)
    X_list.append(ts)

if len(X_list) == 0:
    raise ValueError("No valid time series were found for CAP analysis.")

X_cap = np.vstack(X_list)   # shape: (N_frames_kept, 24)
print("All frames stacked:", X_cap.shape)

# =========================
# 2) Davies-Bouldin index
# =========================
range_n_clusters = range(2, 15)
db_scores = []

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X_cap)
    score = davies_bouldin_score(X_cap, labels)
    db_scores.append(score)
    print(f"k={k}, DB index={score:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(range_n_clusters), db_scores, marker="o", linestyle="-", linewidth=1.8)
ax.set_xlabel("Number of clusters (k)", fontsize=12)
ax.set_ylabel("Davies-Bouldin Index (lower is better)", fontsize=12)
ax.set_title("Davies-Bouldin vs. k (CAP on scrubbed frames)", fontsize=13)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks(list(range_n_clusters))

# Mark the best k
best_idx = int(np.argmin(db_scores))
best_k = list(range_n_clusters)[best_idx]
best_val = db_scores[best_idx]
ax.scatter([best_k], [best_val], s=80, zorder=3)

# Optional annotation
# ax.annotate(
#     f"best k={best_k}\nDB={best_val:.3f}",
#     (best_k, best_val),
#     textcoords="offset points",
#     xytext=(10, -15)
# )

plt.tight_layout()
plt.savefig("davies_bouldin_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
### <h1 style="font-size:36px;">Step 1: XXXX </h1>

<h2 style="font-size:24px;">HMM </h2>

In [ ]:
import numpy as np
from scipy.stats import zscore
from hmmlearn import hmm
from collections import defaultdict

# -----------------------------
pre_path  = "path/to//time_series_list_24_preCondRest.npy"
post_path = "path/to//time_series_list_24_postCondRest.npy"

time_series_list_pre  = list(np.load(pre_path,  allow_pickle=True))   # list of (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))   # list of (T_post_i, 24)

# 
n_pre, n_post = len(time_series_list_pre), len(time_series_list_post)
n_subj = min(n_pre, n_post)
time_series_list_pre  = time_series_list_pre[:n_subj]
time_series_list_post = time_series_list_post[:n_subj]

# 
for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, f"Subject {i} pre shape={ts_pre.shape}"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, f"Subject {i} post shape={ts_post.shape}"

In [ ]:
import numpy as np
n_subj = len(time_series_list_pre)
subject_ids = list(range(n_subj))  
subject_ts_dict_pre = {i: np.asarray(ts) for i, ts in zip(subject_ids, time_series_list_pre)}
subject_ts_dict_post = {i: np.asarray(ts) for i, ts in zip(subject_ids, time_series_list_post)}

In [ ]:
import numpy as np
from scipy.stats import zscore
from hmmlearn import hmm
from collections import defaultdict

# -----------------------------
# 1) Prepare HMM training data
# -----------------------------
if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

n_subj = len(time_series_list_pre)

X_list = []            # list of arrays to be concatenated for HMM input
lengths = []           # sequence length for each segment
subj_idx_per_seq = []  # subject index for each sequence
phase_per_seq = []     # 'pre' or 'post'

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    ts_z = np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)
    return ts_z

for s in range(n_subj):
    ts_pre = np.asarray(time_series_list_pre[s], dtype=float)
    ts_post = np.asarray(time_series_list_post[s], dtype=float)

    if ts_pre.ndim != 2 or ts_pre.shape[1] != 24:
        raise ValueError(f"Subject {s} pre shape={ts_pre.shape}, expected (*, 24)")
    if ts_post.ndim != 2 or ts_post.shape[1] != 24:
        raise ValueError(f"Subject {s} post shape={ts_post.shape}, expected (*, 24)")

    ts_pre_z = safe_zscore(ts_pre, axis=0)    
    ts_post_z = safe_zscore(ts_post, axis=0)  

    if ts_pre_z.shape[0] > 0:
        X_list.append(ts_pre_z)
        lengths.append(ts_pre_z.shape[0])
        subj_idx_per_seq.append(s)
        phase_per_seq.append("pre")

    if ts_post_z.shape[0] > 0:
        X_list.append(ts_post_z)
        lengths.append(ts_post_z.shape[0])
        subj_idx_per_seq.append(s)
        phase_per_seq.append("post")

if len(X_list) == 0:
    raise ValueError("No valid sequences were found for HMM training.")


X = np.vstack(X_list)   

# -----------------------------
# 2) Train HMM
# -----------------------------
k_states = 5

model = hmm.GaussianHMM(
    n_components=k_states,
    covariance_type="diag",
    n_iter=200,
    random_state=1,
    verbose=True
).fit(X, lengths)

state_seq = model.predict(X, lengths)

# -----------------------------
# 3) Split state sequence back into subject x phase
# -----------------------------
state_index_dict_pre = defaultdict(list)
state_index_dict_post = defaultdict(list)

offset = 0
for seq_len, subj_idx, phase in zip(lengths, subj_idx_per_seq, phase_per_seq):
    seq_states = state_seq[offset: offset + seq_len].tolist()
    offset += seq_len

    if phase == "pre":
        state_index_dict_pre[subj_idx].extend(seq_states)
    else:
        state_index_dict_post[subj_idx].extend(seq_states)

print(f"Number of subjects: {n_subj}")
print(
    f"Example subject 0: "
    f"pre length={len(state_index_dict_pre[0])}, "
    f"post length={len(state_index_dict_post[0])}"
)

<h1 style="font-size:24px;">OCR </h1>

In [ ]:
####################### Calculate the proportion #################################

In [ ]:
import pandas as pd
import numpy as np

def compute_subject_ocr(state_index_dict, k_clusters):
    """
    Compute subject-level state occupancy proportions.

    Supported input formats:
      1) subject -> list[int]
      2) subject -> {task_name: list[int], ...}
    """
    rows = []
    for subj, val in state_index_dict.items():
        all_states = []

        if isinstance(val, dict):                 # subject -> task -> list
            for states in val.values():
                all_states.extend(states)
        elif isinstance(val, (list, np.ndarray)): # subject -> list
            all_states.extend(val)
        else:
            raise TypeError(f"Unsupported value type for subject {subj}: {type(val)}")

        if len(all_states) == 0:
            continue

        counts = np.bincount(np.asarray(all_states, dtype=int), minlength=k_clusters)
        props  = counts / counts.sum()
        rows.append([subj, *props])

    cols = ["Subject"] + [f"State{i}_prop" for i in range(k_clusters)]
    return pd.DataFrame(rows, columns=cols)

# Usage
df_pre  = compute_subject_ocr(state_index_dict_pre,  k_clusters=k_states)
df_post = compute_subject_ocr(state_index_dict_post, k_clusters=k_states)

In [ ]:
# --------------------------------------------------
# Paired t-test and FDR correction across states
# --------------------------------------------------

import numpy as np
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection

# Number of clusters (states)
k_clusters = k_states

# Compute subject-level occupancy rate (OCR)
df_pre  = compute_subject_ocr(state_index_dict_pre,  k_clusters=k_clusters)
df_post = compute_subject_ocr(state_index_dict_post, k_clusters=k_clusters)

# Convert to numpy arrays (remove subject column)
pre_array  = df_pre.iloc[:, 1:].to_numpy()
post_array = df_post.iloc[:, 1:].to_numpy()

# Paired t-tests for each state
t_vals = []
p_vals = []

for i in range(pre_array.shape[1]):
    t_stat, p_val = ttest_rel(pre_array[:, i], post_array[:, i])
    t_vals.append(t_stat)
    p_vals.append(p_val)

# FDR correction across states
reject, p_vals_corrected = fdrcorrection(p_vals, alpha=0.05)

# Display results
for i, (t_stat, p_corr, sig) in enumerate(zip(t_vals, p_vals_corrected, reject)):
    marker = "*" if sig else ""
    print(f"State {i}: t = {t_stat:.3f}, p_FDR = {p_corr:.6f} {marker}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_block_state_counts(state_index_dict, target_state=0, window_size=5):
    """
    Compute block-wise counts of a target state for each subject.

    For each subject, the state sequence is divided into non-overlapping
    windows (blocks), and the number of occurrences of the target state
    is counted within each block.

    Parameters
    ----------
    state_index_dict : dict
        Mapping from subject ID to state sequence. Supported formats:
        1) subject -> list[int]
        2) subject -> np.ndarray
        3) subject -> dict of sequences
    target_state : int, default=0
        State label to count.
    window_size : int, default=5
        Number of TRs per block.

    Returns
    -------
    dict
        Dictionary mapping each subject to a 1D array of block-wise counts.
    """
    block_counts_dict = {}

    for subj, val in state_index_dict.items():
        if isinstance(val, dict):
            all_states = []
            for seq in val.values():
                all_states.extend(seq)
            seq = np.asarray(all_states, dtype=int)
        else:
            seq = np.asarray(val, dtype=int)

        T = len(seq)
        if T < window_size:
            block_counts_dict[subj] = np.array([], dtype=int)
            continue

        indicator = (seq == target_state).astype(int)
        n_blocks = T // window_size

        trimmed = indicator[: n_blocks * window_size]
        blocks = trimmed.reshape(n_blocks, window_size)
        counts = blocks.sum(axis=1)

        block_counts_dict[subj] = counts

    return block_counts_dict


def compute_group_block_curve(state_index_dict, target_state=0, window_size=5):
    """
    Compute the group-level block-wise curve for a target state.

    The group curve is obtained by averaging the block-wise counts across
    subjects for each block index. Only subjects with sufficient sequence
    length contribute to each block.

    Returns
    -------
    block_counts_dict : dict
        Subject-level block-wise counts.
    group_curve : np.ndarray
        Group-average block-wise counts.
    """
    block_counts_dict = compute_block_state_counts(
        state_index_dict,
        target_state=target_state,
        window_size=window_size
    )

    valid_blocks = [arr for arr in block_counts_dict.values() if len(arr) > 0]

    if len(valid_blocks) == 0:
        return block_counts_dict, np.array([])

    max_len = max(len(arr) for arr in valid_blocks)
    group_curve = []

    for i in range(max_len):
        vals = [arr[i] for arr in valid_blocks if len(arr) > i]
        group_curve.append(np.mean(vals))

    group_curve = np.asarray(group_curve)
    return block_counts_dict, group_curve


# Example usage
target_state = 0
window_size = 30

block_post_dict, group_curve_post = compute_group_block_curve(
    state_index_dict_post,
    target_state=target_state,
    window_size=window_size
)

print("Length of group_curve_post (number of blocks):", len(group_curve_post))
print("Counts in the first 10 blocks:", group_curve_post[:10])

x = np.arange(1, len(group_curve_post) + 1)

plt.figure(figsize=(10, 4))
plt.plot(x, group_curve_post, marker="o")
plt.xlabel(f"Block index (each = {window_size} TRs)")
plt.ylabel(f"Count of state {target_state} within block")
plt.title(
    f"Group-level block count curve for state {target_state} "
    f"(post, window = {window_size} TRs)"
)
plt.grid(True)
plt.tight_layout()
plt.show()

<h1 style="font-size:28px;">DT</h1>

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def _accumulate_dwell_for_sequence(seq, k_clusters, collector):
    if seq is None or len(seq) == 0:
        return

    s = np.asarray(seq, dtype=int).tolist()

    curr = s[0]
    dwell = 1
    for x in s[1:]:
        if x == curr:
            dwell += 1
        else:
            collector[curr].append(dwell)
            curr = x
            dwell = 1
    collector[curr].append(dwell)

def compute_dwell_times(state_index_dict, k_clusters):
    """
    Compute dwell times for each subject and each state.

    Supported input formats:
      A) single-level: {subject: [state, state, ...]}
      B) nested:       {subject: {task: [state, state, ...]}}

    Returns
    -------
    dwell_times_dict : dict
        dwell_times_dict[subject][state] = [dwell1, dwell2, ...]
    """
    dwell_times_dict = defaultdict(lambda: defaultdict(list))

    for subj_id, val in state_index_dict.items():
        if isinstance(val, dict):
            for task, seq in val.items():
                _accumulate_dwell_for_sequence(seq, k_clusters, dwell_times_dict[subj_id])
        elif isinstance(val, (list, np.ndarray)):
            _accumulate_dwell_for_sequence(val, k_clusters, dwell_times_dict[subj_id])
        else:
            raise TypeError(f"Unsupported value type for subject {subj_id}: {type(val)}")

    return dwell_times_dict

def summarize_dwell(dwell_times_dict, k_clusters):
    state_subject_means = {s: [] for s in range(k_clusters)}

    for subj, per_state in dwell_times_dict.items():
        for s in range(k_clusters):
            dws = per_state.get(s, [])
            if len(dws) > 0:
                state_subject_means[s].append(float(np.mean(dws)))
            else:
                state_subject_means[s].append(0.0)

    print("=== Mean dwell time for each state ===")
    for s in range(k_clusters):
        arr = np.asarray(state_subject_means[s], dtype=float)
        if arr.size > 0:
            print(f"State {s}: mean = {arr.mean():.2f} ± {arr.std(ddof=1):.2f}, N_subjects = {arr.size}")
        else:
            print(f"State {s}: no data")

    return state_subject_means

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection

k_clusters = k_states

dwell_pre = compute_dwell_times(state_index_dict_pre, k_clusters)
dwell_post = compute_dwell_times(state_index_dict_post, k_clusters)

print("\nPre:")
dwell_pre_avg = summarize_dwell(dwell_pre, k_clusters)

print("\nPost:")
dwell_post_avg = summarize_dwell(dwell_post, k_clusters)


def dwell_to_df(dwell_times_dict, k_clusters):
    rows = []
    for subj, per_state in dwell_times_dict.items():
        row = {"Subject": subj}
        for s in range(k_clusters):
            dws = per_state.get(s, [])
            row[f"State{s}_mean_dwell"] = float(np.mean(dws)) if len(dws) > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


df_dwell_pre = dwell_to_df(dwell_pre, k_clusters)
df_dwell_post = dwell_to_df(dwell_post, k_clusters)


# Paired t-tests across states
t_vals = []
p_vals = []
state_cols = df_dwell_pre.columns[1:]

for col in state_cols:
    tval, pval = ttest_rel(df_dwell_pre[col], df_dwell_post[col])
    t_vals.append(tval)
    p_vals.append(pval)

# FDR correction across states
reject, p_vals_fdr = fdrcorrection(p_vals, alpha=0.05)

# Print results
for col, tval, pval, pval_fdr, sig in zip(state_cols, t_vals, p_vals, p_vals_fdr, reject):
    marker = "*" if sig else ""
    print(f"{col}: t = {tval:.3f}, p = {pval:.4f}, p_FDR = {pval_fdr:.4f} {marker}")

<h1 style="font-size:24px;">Transition Matrix </h1>

In [ ]:
######################## Transition Matrix ###########################

In [ ]:
import numpy as np

def compute_transition_matrices(state_index_dict, k_clusters):
    """
    Compute a k_clusters x k_clusters transition probability matrix
    for each subject.

    Input
    -----
    state_index_dict[subject] = [state sequence]

    Output
    ------
    subject_transition_matrices[subject] = transition probability matrix
    """
    subject_transition_matrices = {}

    for subj, state_seq in state_index_dict.items():
        transitions = np.zeros((k_clusters, k_clusters), dtype=int)

        for i in range(1, len(state_seq)):
            prev_state = state_seq[i - 1]
            curr_state = state_seq[i]
            transitions[prev_state, curr_state] += 1

        row_sums = transitions.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            trans_prob = np.divide(transitions, row_sums, where=row_sums != 0)

        subject_transition_matrices[subj] = trans_prob

    return subject_transition_matrices

In [ ]:
k_clusters = k_states

trans_pre = compute_transition_matrices(state_index_dict_pre, k_clusters)
trans_post = compute_transition_matrices(state_index_dict_post, k_clusters)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# --------------------------------------------------
# 1) Paired t-tests on transition probabilities
#    with FDR correction across all k x k cells
# --------------------------------------------------

subject_ids = sorted(set(trans_pre.keys()) & set(trans_post.keys()))
trans_pre_array = np.stack([trans_pre[subj] for subj in subject_ids], axis=0)    # (N, k, k)
trans_post_array = np.stack([trans_post[subj] for subj in subject_ids], axis=0)  # (N, k, k)

raw_p_list = []
t_list = []
index_list = []

for i in range(k_clusters):
    for j in range(k_clusters):
        pre_vals = trans_pre_array[:, i, j]
        post_vals = trans_post_array[:, i, j]
        t_stat, p_val = ttest_rel(pre_vals, post_vals, nan_policy="omit")
        t_list.append(t_stat)
        raw_p_list.append(p_val)
        index_list.append((i, j))

# FDR correction
alpha = 0.05
reject, p_fdr_list = fdrcorrection(raw_p_list, alpha=alpha)

# Store corrected p-values in a matrix
p_fdr_matrix = np.full((k_clusters, k_clusters), np.nan)
annot_matrix = np.empty((k_clusters, k_clusters), dtype=object)

for idx, (i, j) in enumerate(index_list):
    p_fdr = p_fdr_list[idx]
    p_fdr_matrix[i, j] = p_fdr

    if p_fdr < 0.001:
        annot_matrix[i, j] = "***"
    elif p_fdr < 0.01:
        annot_matrix[i, j] = "**"
    elif p_fdr < alpha:
        annot_matrix[i, j] = "*"
    else:
        annot_matrix[i, j] = ""

# Group-level mean difference
diff_matrix = trans_pre_array.mean(axis=0) - trans_post_array.mean(axis=0)

# --------------------------------------------------
# 2) Bubble plot with FDR-corrected significance labels
# --------------------------------------------------

cmap = sns.diverging_palette(220, 20, as_cmap=True)

def bubble_transition_plot(
    diff_matrix,
    annot_matrix=None,
    title="",
    save_name="Trans_bubble_fdr.svg",
    cmap=cmap,
    max_bubble_area=650,
    min_bubble_area=20,
    grid_lw=1.2,
    bubble_edge_lw=0.6
):
    k = diff_matrix.shape[0]

    v = np.max(np.abs(diff_matrix)) if np.max(np.abs(diff_matrix)) > 0 else 1.0
    norm = Normalize(vmin=-v, vmax=v)

    xs, ys = np.meshgrid(np.arange(k), np.arange(k))
    x = xs.ravel()
    y = ys.ravel()
    vals = diff_matrix.ravel()

    absvals = np.abs(vals)
    if absvals.max() == 0:
        sizes = np.full_like(absvals, min_bubble_area, dtype=float)
    else:
        sizes = min_bubble_area + (max_bubble_area - min_bubble_area) * (absvals / absvals.max())

    colors = cmap(norm(vals))

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        x, y,
        s=sizes,
        c=colors,
        edgecolors="white",
        linewidths=bubble_edge_lw
    )

    # Add significance labels
    if annot_matrix is not None:
        for i in range(k):
            for j in range(k):
                label = annot_matrix[i, j]
                if label != "":
                    ax.text(
                        j, i, label,
                        ha="center", va="center",
                        fontsize=11, fontweight="bold", color="black"
                    )

    ax.set_xlim(-0.5, k - 0.5)
    ax.set_ylim(k - 0.5, -0.5)
    ax.set_aspect("equal")

    ax.set_xticks(np.arange(k))
    ax.set_yticks(np.arange(k))

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticks(np.arange(-0.5, k, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax.grid(which="minor", color="#BFBFBF", linestyle="-", linewidth=grid_lw)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_title(title, fontsize=14)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pre - Post Probability Difference")

    plt.tight_layout()
    plt.savefig(save_name, format="svg", dpi=300, bbox_inches="tight")
    plt.show()

# Plot
bubble_transition_plot(
    diff_matrix,
    annot_matrix=annot_matrix,
    title="Pre - Post Transition Difference (FDR-corrected)",
    save_name="Trans_bubble_fdr.svg"
)

# --------------------------------------------------
# 3) Print significant transitions after FDR correction
# --------------------------------------------------

sig_indices_fdr = [(i, j) for (i, j), sig in zip(index_list, reject) if sig]
print(f"Number of significant transitions (FDR-corrected, q < {alpha}): {len(sig_indices_fdr)}")
print("Significant indices (from_state, to_state):", sig_indices_fdr)

for (i, j), t_stat, p_raw, p_fdr, sig in zip(index_list, t_list, raw_p_list, p_fdr_list, reject):
    marker = "*" if sig else ""
    print(f"Transition {i}->{j}: t = {t_stat:.3f}, p = {p_raw:.4f}, p_FDR = {p_fdr:.4f} {marker}")

<h1 style="font-size:24px;">Entropy </h1>

In [ ]:
import numpy as np
import scipy.linalg
from scipy.stats import ttest_rel

# =========================
# Entropy-related functions
# =========================
def as_stochastic(P, axis=1, eps=1e-12):
    """Normalize a matrix into a row-stochastic transition probability matrix."""
    P = np.asarray(P, dtype=float)
    P = np.clip(P, 0.0, 1.0)
    row_sums = P.sum(axis=axis, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return P / row_sums


def local_entropy(P, logbase=2):
    """Compute local entropy for each row of a transition matrix."""
    P = as_stochastic(P)
    log = np.log2 if logbase == 2 else np.log

    L = np.zeros_like(P)
    mask = P > 0
    L[mask] = log(P[mask])

    K = P @ L.T
    entropy_out = -np.diag(K)
    return entropy_out.reshape((P.shape[0], 1))


def stationary_distribution(P):
    """Compute the stationary distribution of a transition matrix."""
    P = as_stochastic(P)
    try:
        w, vl = scipy.linalg.eig(P, left=True, right=False)
        idx = np.argmin(np.abs(w - 1))
        v = np.real(vl[:, idx])
        mu = np.abs(v)
        s = mu.sum()
        if s == 0:
            raise RuntimeError
        mu /= s
    except Exception:
        mu = np.ones(P.shape[0]) / P.shape[0]
        for _ in range(2000):
            mu_new = mu @ P
            if np.linalg.norm(mu_new - mu, ord=1) < 1e-12:
                break
            mu = mu_new
    return mu


def trajectory_entropy(P, logbase=2, ridge=1e-8):
    """
    Compute the trajectory entropy matrix H for a transition matrix P.
    """
    P = as_stochastic(P)
    n = P.shape[0]
    mu = stationary_distribution(P)
    A = np.tile(mu, (n, 1))
    l_entropy = local_entropy(P, logbase=logbase)

    H_star = np.tile(l_entropy, (1, n))
    entropy_rate = float(mu @ l_entropy.flatten())

    mu_safe = mu.copy()
    mu_safe[mu_safe == 0] = 1e-12
    H_delta = np.diag(entropy_rate / mu_safe)

    M = np.eye(n) - P + A
    try:
        Minv = np.linalg.inv(M)
    except np.linalg.LinAlgError:
        Minv = np.linalg.inv(M + ridge * np.eye(n))

    K = Minv @ (H_star - H_delta)
    K_diag = np.diag(K).reshape(-1, 1)
    K_tilda = np.tile(K_diag.T, (n, 1))

    H = K - K_tilda + H_delta
    return H


def normalized_trajectory_entropy(P, logbase=2):
    """
    Compute the normalized trajectory entropy matrix.
    H_norm = H / entropy_rate
    """
    H = trajectory_entropy(P, logbase=logbase)

    mu = stationary_distribution(P)
    h_row = local_entropy(P, logbase=logbase).flatten()
    H_rate = float(mu @ h_row)

    if H_rate == 0:
        return np.zeros_like(H)

    return H / H_rate


# =========================
# Convert dict to aligned arrays
# =========================
def stack_subject_transition(trans_pre, trans_post):
    common = sorted(set(trans_pre.keys()) & set(trans_post.keys()))
    P_pre = np.stack([as_stochastic(trans_pre[s]) for s in common], axis=0)
    P_post = np.stack([as_stochastic(trans_post[s]) for s in common], axis=0)
    return P_pre, P_post, common


P_pre, P_post, subject_ids = stack_subject_transition(trans_pre, trans_post)
N, k, _ = P_pre.shape
print(f"Stacked: N = {N}, k = {k}")

# =========================
# Subject-level scalar trajectory entropy: mean(H)
# =========================
pre_scalar = np.empty(N, dtype=float)
post_scalar = np.empty(N, dtype=float)

for i in range(N):
    Hpre = trajectory_entropy(P_pre[i], logbase=2)
    Hpost = trajectory_entropy(P_post[i], logbase=2)
    pre_scalar[i] = Hpre.mean()
    post_scalar[i] = Hpost.mean()

# Paired t-test on the scalar mean(H)
tval, pval = ttest_rel(post_scalar, pre_scalar)
print(f"[Scalar mean(H)] paired t-test: t = {tval:.3f}, p = {pval:.4g}")

# =========================
# Subject-level trajectory entropy matrices: H (N, k, k)
# =========================
# pre_H = np.stack([trajectory_entropy(P_pre[i], logbase=2) for i in range(N)], axis=0)
# post_H = np.stack([trajectory_entropy(P_post[i], logbase=2) for i in range(N)], axis=0)

pre_H = np.stack([normalized_trajectory_entropy(P_pre[i]) for i in range(N)], axis=0)
post_H = np.stack([normalized_trajectory_entropy(P_post[i]) for i in range(N)], axis=0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def plot_t_matrix_bubble(
    pre_H,
    post_H,
    title="Paired t-statistic Matrix",
    save_name="T_matrix.svg",
    cmap=None,
    max_size=650,
    min_size=30
):
    """
    Plot a bubble matrix of paired t-statistics comparing pre_H and post_H.

    Parameters
    ----------
    pre_H : np.ndarray
        Array of shape (N, k, k) for the pre condition.
    post_H : np.ndarray
        Array of shape (N, k, k) for the post condition.
    title : str
        Figure title.
    save_name : str
        Output file name for the saved figure.
    cmap : matplotlib colormap, optional
        Colormap used for the bubble colors.
    max_size : float
        Maximum bubble size.
    min_size : float
        Minimum bubble size.

    Returns
    -------
    t_mat : np.ndarray
        Matrix of paired t-statistics.
    p_mat : np.ndarray
        Matrix of raw p-values.
    """
    if cmap is None:
        cmap = sns.diverging_palette(220, 20, as_cmap=True)

    N, k, _ = pre_H.shape

    t_mat = np.zeros((k, k))
    p_mat = np.zeros((k, k))

    # Compute paired t-tests
    for i in range(k):
        for j in range(k):
            t, p = ttest_rel(pre_H[:, i, j], post_H[:, i, j], nan_policy="omit")
            t_mat[i, j] = t
            p_mat[i, j] = p

    # Color scaling
    vmax = np.percentile(np.abs(t_mat), 98)
    vmax = vmax if vmax > 0 else 1
    norm = Normalize(vmin=-vmax, vmax=vmax)

    # Coordinates
    xs, ys = np.meshgrid(np.arange(k), np.arange(k))
    x = xs.ravel()
    y = ys.ravel()
    vals = t_mat.ravel()

    # Bubble sizes
    absvals = np.abs(vals)
    if absvals.max() == 0:
        sizes = np.full_like(absvals, min_size, dtype=float)
    else:
        sizes = min_size + (max_size - min_size) * (absvals / absvals.max())

    colors = cmap(norm(vals))

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        x, y,
        s=sizes,
        c=colors,
        edgecolors="white",
        linewidths=0.7
    )

    ax.set_xlim(-0.5, k - 0.5)
    ax.set_ylim(k - 0.5, -0.5)
    ax.set_aspect("equal")

    ax.set_xticks(np.arange(k))
    ax.set_yticks(np.arange(k))
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(title, fontsize=14)

    # Grid
    ax.set_xticks(np.arange(-0.5, k, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax.grid(which="minor", color="#BFBFBF", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("t statistic")

    plt.tight_layout()
    plt.savefig(save_name, format="svg", dpi=300, bbox_inches="tight")
    plt.show()

    return t_mat, p_mat